In [ ]:
from pyspark.sql import SparkSession

# ============================================================
# Paths
# ============================================================

RAW_PATH    = "s3a://rawload"
BRONZE_PATH = "s3a://bronzeload"
SILVER_PATH = "s3a://silverload"
GOLD_PATH   = "s3a://goldload"

RAW_FILE = (
    f"{RAW_PATH}/agesexbyethnicgroup/Data8277.csv"
)

HUDI_PATH = f"{BRONZE_PATH}/agesexbyethnicgroup"
DELTA_PATH = f"{SILVER_PATH}/agesexbyethnicgroup"

# ============================================================
# Spark
# ============================================================

spark = (
    SparkSession.builder
    .appName("Three Table Format Comparison")

    # --------------------------------------------------------
    # Local Hudi bundle
    # --------------------------------------------------------

    .config(
        "spark.jars",
        r"C:\data\datatool\spark\jars"
        r"\hudi-spark4.0-bundle_2.13-1.2.0.jar"
    )

    # --------------------------------------------------------
    # Delta + Iceberg + Hadoop AWS
    # --------------------------------------------------------

    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.13:4.0.0,"
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
        "org.apache.hadoop:hadoop-aws:3.4.1"
    )

    # --------------------------------------------------------
    # Spark SQL extensions
    # --------------------------------------------------------
    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension,"
        "io.delta.sql.DeltaSparkSessionExtension,"
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        )

    # --------------------------------------------------------
    # Delta catalog
    # --------------------------------------------------------

    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    # --------------------------------------------------------
    # Hudi
    # --------------------------------------------------------

    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )

    .config(
        "spark.kryo.registrator",
        "org.apache.spark.HoodieSparkKryoRegistrar"
    )

    # --------------------------------------------------------
    # Iceberg catalog
    # --------------------------------------------------------

    .config(
        "spark.sql.catalog.ice",
        "org.apache.iceberg.spark.SparkCatalog"
    )

    .config(
        "spark.sql.catalog.ice.catalog-impl",
        "org.apache.iceberg.hadoop.HadoopCatalog"
    )

    .config(
        "spark.sql.catalog.ice.warehouse",
        GOLD_PATH
    )

    # --------------------------------------------------------
    # MinIO / S3A
    # --------------------------------------------------------

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )

    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .getOrCreate()
)

print("Spark :", spark.version)

print(
    "Hadoop:",
    spark.sparkContext._jvm
        .org.apache.hadoop.util.VersionInfo.getVersion()
)

In [ ]:
from pyspark.sql import functions as F

Data8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/Data8277.csv")
)

print("RAW rows:", Data8277.count())

Data8277.printSchema()

Data8277.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

DimenLookupAge8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/DimenLookupAge8277.csv")
)

print("RAW rows:", DimenLookupAge8277.count())

DimenLookupAge8277.printSchema()

DimenLookupAge8277.show(20, truncate=False)

In [ ]:
from pyspark.sql import functions as F

DimenLookupArea8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/DimenLookupArea8277.csv")
)

print("RAW rows:", DimenLookupArea8277.count())

DimenLookupArea8277.printSchema()

DimenLookupArea8277.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

DimenLookupEthnic8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/DimenLookupEthnic8277.csv")
)

print("RAW rows:", DimenLookupEthnic8277.count())

DimenLookupEthnic8277.printSchema()

DimenLookupEthnic8277.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

DimenLookupSex8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/DimenLookupSex8277.csv")
)

print("RAW rows:", DimenLookupSex8277.count())

DimenLookupSex8277.printSchema()

DimenLookupSex8277.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

DimenLookupYear8277 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RAW_PATH + "/agesexbyethnicgroup/DimenLookupYear8277.csv")
)

print("RAW rows:", DimenLookupYear8277.count())

DimenLookupYear8277.printSchema()

DimenLookupYear8277.show(5, truncate=False)

In [ ]:
spark.sql("show catalogs").show()
spark.sql("show namespaces in ice").show()

In [ ]:
from pyspark.sql import functions as F

Data8277.writeTo("ice.gold.fact_agesexbyethnicgroup") \
    .using("iceberg") \
    .partitionedBy("year", "area") \
    .tableProperty("write.format.default", "avro") \
    .createOrReplace()

In [ ]:
from pyspark.sql import functions as F
tbl_fact_agesexbyethnicgroup = spark.table("ice.gold.fact_agesexbyethnicgroup")
non_numeric_df = tbl_fact_agesexbyethnicgroup.filter(
    F.col("count").try_cast("bigint").isNull() & 
    F.col("count").isNotNull()
)
non_numeric_df.count()

In [ ]:
spark.sql("""
    UPDATE ice.gold.fact_agesexbyethnicgroup
    SET count = 0
    WHERE try_cast(count AS double) IS NULL 
      AND count IS NOT NULL
""")

In [ ]:
# spark.sql("""
#     ALTER TABLE ice.gold.fact_agesexbyethnicgroup 
#     ALTER COLUMN count TYPE bigint
# """)

from pyspark.sql import functions as F

# 1. Read existing data and cast the field to target type
cast_df = spark.table("ice.gold.fact_agesexbyethnicgroup") \
               .withColumn("count", F.col("count").cast("double"))

# 2. Re-create or Replace table layout atomically 
cast_df.writeTo("ice.gold.fact_agesexbyethnicgroup") \
       .using("iceberg") \
       .partitionedBy("year", "area") \
       .createOrReplace()

fact_agesexbyethnicgroup = spark.table("ice.gold.fact_agesexbyethnicgroup").alias("fact_agesexbyethnicgroup")

In [ ]:
from pyspark.sql import functions as F

DimenLookupAge8277.writeTo("ice.gold.dim_age") \
    .using("iceberg") \
    .tableProperty("write.format.default", "orc") \
    .createOrReplace()

dim_age = spark.table("ice.gold.dim_age").alias("dim_age")

In [ ]:
from pyspark.sql import functions as F

DimenLookupArea8277.writeTo("ice.gold.dim_area") \
    .using("iceberg") \
    .tableProperty("write.format.default", "orc") \
    .createOrReplace()

dim_area = spark.table("ice.gold.dim_area").alias("dim_area")

In [ ]:
from pyspark.sql import functions as F

DimenLookupEthnic8277.writeTo("ice.gold.dim_ethnic") \
    .using("iceberg") \
    .tableProperty("write.format.default", "orc") \
    .createOrReplace()

dim_ethnic = spark.table("ice.gold.dim_ethnic").alias("dim_ethnic")

In [ ]:
from pyspark.sql import functions as F

DimenLookupSex8277.writeTo("ice.gold.dim_sex") \
    .using("iceberg") \
    .tableProperty("write.format.default", "orc") \
    .createOrReplace()

dim_sex = spark.table("ice.gold.dim_sex").alias("dim_sex")

In [ ]:
from pyspark.sql import functions as F

DimenLookupYear8277.writeTo("ice.gold.dim_year") \
    .using("iceberg") \
    .tableProperty("write.format.default", "orc") \
    .createOrReplace()

dim_year = spark.table("ice.gold.dim_year").alias("dim_year")

In [ ]:
dim_ethnic.show(5, truncate=False)
dim_area.show(5, truncate=False)
fact_agesexbyethnicgroup.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

all_output = fact_agesexbyethnicgroup.join(
    dim_age,F.col("fact_agesexbyethnicgroup.Age") == F.col("dim_age.Code"),"left") \
    .join(dim_area,F.col("fact_agesexbyethnicgroup.Area") == F.col("dim_area.Code"),"left") \
    .join(dim_ethnic,F.col("fact_agesexbyethnicgroup.Ethnic") == F.col("dim_ethnic.Code"),"left") \
    .join(dim_sex,F.col("fact_agesexbyethnicgroup.Sex") == F.col("dim_sex.Code"),"left") \
    .join(dim_year,F.col("fact_agesexbyethnicgroup.Year") == F.col("dim_year.Code"),"left") \
    .select(
        F.col("fact_agesexbyethnicgroup.count").alias("count"),
        F.col("dim_area.Description").alias("area_Description"),
        F.col("dim_age.Description").alias("age_Description"),
        F.col("dim_ethnic.Description").alias("ethnicgroup_Description"),
        F.col("dim_sex.Description").alias("sex_Description"),
        F.col("dim_year.Description").alias("year_Description")
    )
all_output.show(5, truncate=False)

In [81]:
spark.stop()